In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import string

In [ ]:
#NASDAQ-100 (NDX) Historical Data from 25-11-2015 to 24-11-2025
data = pd.read_csv("C:/Users/apkar/Downloads\HistoricalData_1764072396542.csv")

In [ ]:
data['Year'] = [int(i.split('/')[2]) for i in list(data['Date'])]
data['Date'] = pd.to_datetime(data['Date'], format='%m/%d/%Y')
data = data.sort_values('Date').reset_index(drop=True)
data['Percent change'] = np.where(
    data['Open'] != 0,
    (data['Close/Last'] - data['Open']) * 100 / data['Open'],
    0
)
data['Vol'] = abs(data['High'] - data['Low'])

In [ ]:
# Setting State cuttofs and parameters

def categorize_percentiles(series):
    q33 = series.quantile(0.33)
    q66 = series.quantile(0.66)

    def label(x):
        if x <= q33:
            return "Low"
        elif q33 < x <= q66:
            return "Medium"
        else:
            return "High"
    return series.apply(label)
def categorize_percent_change(series):
    pos = series[series > 0]
    neg = series[series < 0]

    # percentiles for positive values
    qpos50 = pos.quantile(0.50)
    qpos80 = pos.quantile(0.80)

    # percentiles for negative values
    qneg50 = neg.quantile(0.50)
    qneg20 = neg.quantile(0.20)

    def label(x):
        if x == 0:
            return "Consistent"

        # Positive values
        if x > 0:
            if x <= qpos50:
                return "Increase"
            else:
                return "High Increase"

        # Negative values
        else:
            if x >= qneg50:
                return "Decrease"
            else:
                return "High Decrease"

    return series.apply(label)
cols_to_categorize = ["Open", "Close/Last", "High", "Low", "Vol"]

for col in cols_to_categorize:
    data[col + "_Category"] = categorize_percentiles(data[col])

# Special handling for Percent change
data["PercentChange_Category"] = categorize_percent_change(data["Percent change"])




In [ ]:
# --Defining  Ultimate states

#  Choose columns to combine into a state
state_cols = [
    "Open_Category",
    "High_Category",
    "Low_Category",
    "Close/Last_Category",
    "Vol_Category",
    "PercentChange_Category"
]

#  Build the raw state description for each row
data["State_Desc"] = data[state_cols].agg("|".join, axis=1)

#  Get all unique states
unique_states = data["State_Desc"].unique()

# Assign letters (A, B, C, ... AA, AB... if needed)
letters = []
alphabet = list(string.ascii_uppercase)

# Support more than 26 states
for i in range(len(unique_states)):
    if i < 26:
        letters.append(alphabet[i])
    else:
        # For i >= 26 → AA, AB, AC ...
        letters.append(alphabet[(i//26)-1] + alphabet[i % 26])

#  Build mapping dictionary
state_mapping = dict(zip(unique_states, letters))

#  Apply final state letters to dataset
data["State"] = data["State_Desc"].map(state_mapping)

# OPTIONAL: Create human-readable explanations
explanations = {letters[i]: unique_states[i] for i in range(len(unique_states))}


In [133]:
test = data[data['Year']> 2023]
train = data[data['Year']<=2023]

In [ ]:
# 1. Shift the state column to get next day's state
train['Next_State'] = train['State'].shift(-1)

# Remove last row (no transition out of the final state)
train_clean = train.dropna(subset=['Next_State'])

# 2. Count transitions
transition_counts = (
    train_clean.groupby(['State', 'Next_State'])
               .size()
               .reset_index(name='Count')
)

# 3. Convert counts into probabilities
# Divide each State row by the total outgoing transitions of that State
transition_counts['Probability'] = (
    transition_counts['Count'] /
    transition_counts.groupby('State')['Count'].transform('sum')
)

# 4. Rename the columns as requested
Transition_probability = transition_counts.rename(
    columns={
        'State': 'State1',
        'Next_State': 'State2'
    }
)[['State1', 'State2', 'Probability']]
train['Next_State'] = train['State'].shift(-1)


C:\Users\apkar\AppData\Local\Temp\ipykernel_36888\1870575689.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['Next_State'] = train['State'].shift(-1)


In [109]:
# Count how many times each state appears in the 'State' column
state_counts = train['State'].value_counts().reset_index()
state_counts.columns = ['State', 'Count']

print(state_counts)


   State  Count
0      C    257
1      A    200
2     AG    139
3     AH    139
4      E    101
..   ...    ...
56    AN      1
57    BA      1
58    BB      1
59    BH      1
60    BI      1

[61 rows x 2 columns]


In [110]:
from itertools import permutations,combinations

In [ ]:
# Finding the correlation between Every feature categories
from scipy.stats import chi2_contingency
columns = ['Open_Category', 'Close/Last_Category', 'Vol_Category', 'PercentChange_Category']
col = list(combinations(columns,2))
for i in col:
    a,b = i
    table = pd.crosstab(train[a], train[b])
    chi2, p, dof, expected = chi2_contingency(table)
    n = table.sum().sum()

    cramers_v = np.sqrt(chi2 / (n * (min(table.shape)-1)))
    print(f"Cramér's V for {a} and {b} is ", cramers_v)

Cramér's V for Open_Category and Close/Last_Category is  0.9750525852110238
Cramér's V for Open_Category and Vol_Category is  0.4327662849813064
Cramér's V for Open_Category and PercentChange_Category is  0.12733041250613703
Cramér's V for Close/Last_Category and Vol_Category is  0.4307302039991387
Cramér's V for Close/Last_Category and PercentChange_Category is  0.12561554603328337
Cramér's V for Vol_Category and PercentChange_Category is  0.3720416971149674


In [114]:
for i in columns:
    train[f'Next_State_{i}'] = train[i].shift(-1)

    # Build transition matrix
    transition_matrix = pd.crosstab(train[i], train[f'Next_State_{i}'], normalize='index')

    print(transition_matrix)


Next_State_Open_Category      High       Low    Medium
Open_Category                                         
High                      0.978836  0.000000  0.021164
Low                       0.000000  0.985542  0.014458
Medium                    0.010843  0.013253  0.975904
Next_State_Close/Last_Category      High       Low    Medium
Close/Last_Category                                         
High                            0.978836  0.000000  0.021164
Low                             0.000000  0.981928  0.018072
Medium                          0.010843  0.016867  0.972289
Next_State_Vol_Category      High       Low    Medium
Vol_Category                                         
High                     0.649541  0.009174  0.341284
Low                      0.013333  0.815758  0.170909
Medium                   0.269461  0.218563  0.511976
Next_State_PercentChange_Category  Consistent  Decrease  High Decrease  \
PercentChange_Category                                                   
Co

C:\Users\apkar\AppData\Local\Temp\ipykernel_36888\1831437521.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[f'Next_State_{i}'] = train[i].shift(-1)
C:\Users\apkar\AppData\Local\Temp\ipykernel_36888\1831437521.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train[f'Next_State_{i}'] = train[i].shift(-1)
C:\Users\apkar\AppData\Local\Temp\ipykernel_36888\1831437521.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_inde

In [ ]:
# 1. Create a dictionary for quick access: for each State1, list of possible next states and their probabilities
transition_dict = {}
for state in Transition_probability['State1'].unique():
    subset = Transition_probability[Transition_probability['State1'] == state]
    transition_dict[state] = (subset['State2'].values, subset['Probability'].values)

# 2. Function to predict next state by picking the highest probability transition
def predict_next_state(current_state):
    if current_state not in transition_dict:
        return None  # or some default/fallback
    next_states, probs = transition_dict[current_state]
    return next_states[np.argmax(probs)]  # choose the most likely next state

# 3. Apply prediction to the test dataset
test['Predicted_Next_State'] = test['State'].apply(predict_next_state)

# 4. Get the actual next state from test data (shifted)
test['Actual_Next_State'] = test['State'].shift(-1)

# Drop last row with NaN actual next state
test_clean = test.dropna(subset=['Actual_Next_State'])

In [ ]:
#  ---Accuracy.---
accuracy = (test_clean['Predicted_Next_State'] == test_clean['Actual_Next_State']).mean()

print(f"Prediction accuracy: {accuracy:.4f}")

Prediction accuracy: 0.2206


In [ ]:
# Create a dictionary from Transition_probability
transition_dict = {
    state: {"next_states": group['State2'].values, "probabilities": group['Probability'].values}
    for state, group in Transition_probability.groupby('State1')
}

# Function to predict next state
def predict_next_state(current_state):
    if current_state not in transition_dict:
        return np.nan  # or current_state if you prefer
    choices = transition_dict[current_state]["next_states"]
    probs = transition_dict[current_state]["probabilities"]
    return np.random.choice(choices, p=probs)

# Apply to train dataset
train['Predicted_Next_State'] = train['State'].apply(predict_next_state)



In [ ]:
train['Month'] = [ str(i).split('-')[1] for i in list(train['Date'])]
test['Month'] = [ str(i).split('-')[1] for i in list(test['Date'])]

In [124]:
# Descion tree model 
cols = [ 'Close/Last', 'Open', 'High', 'Low', 'Year', 'Percent change',
       'Vol','Month', 'Next_State']
dt_train = train[[ 'Close/Last', 'Open', 'High', 'Low', 'Year', 'Percent change',
       'Vol','Month', 'Next_State']]
dt_test = test[[ 'Close/Last', 'Open', 'High', 'Low', 'Year', 'Percent change',
       'Vol','Month','Actual_Next_State']]

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize encoder

le = LabelEncoder()

# Encode the column

dt_train["Next_State"] = le.fit_transform(dt_train["Next_State"])
dt_test['Actual_Next_State'] = le.transform(dt_test['Actual_Next_State'])
dt_train = dt_train.sample(frac=1).reset_index(drop=True)
dt_test = dt_test.sample(frac=1).reset_index(drop=True)
xtr,ytr = dt_train.drop('Next_State',axis = 1),dt_train['Next_State']
xts,yts = dt_test.drop('Actual_Next_State',axis = 1),dt_test['Actual_Next_State']

from sklearn.ensemble import RandomForestClassifier 
from sklearn.metrics import accuracy_score,classification_report

# Ran Grid search

model = RandomForestClassifier(n_estimators=400,min_samples_split=4,min_samples_leaf=9,max_depth=23,random_state=42,n_jobs=-1)
model.fit(xtr,ytr)


In [129]:
y_pred = model.predict(xts)

# Evaluation
print("Accuracy:", accuracy_score(yts, y_pred))
print("\nClassification Report:\n", classification_report(yts, y_pred))

Accuracy: 0.1719077568134172

Classification Report:
               precision    recall  f1-score   support

          18       0.00      0.00      0.00        26
          20       0.00      0.00      0.00         3
          21       0.12      0.37      0.18        51
          22       0.20      0.43      0.27       102
          25       0.09      0.07      0.08        75
          26       0.00      0.00      0.00         9
          30       0.00      0.00      0.00        54
          31       0.29      0.13      0.18       104
          32       0.00      0.00      0.00         2
          33       0.00      0.00      0.00        50
          61       0.00      0.00      0.00         1

    accuracy                           0.17       477
   macro avg       0.06      0.09      0.07       477
weighted avg       0.13      0.17      0.13       477



c:\Users\apkar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\apkar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\apkar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [96]:
y_pred = model.predict(xts)

# Evaluation
print("Accuracy:", accuracy_score(yts, y_pred))
print("\nClassification Report:\n", classification_report(yts, y_pred))

Accuracy: 0.1740041928721174

Classification Report:
               precision    recall  f1-score   support

          18       0.00      0.00      0.00        26
          20       0.00      0.00      0.00         3
          21       0.11      0.29      0.16        51
          22       0.20      0.50      0.28       102
          25       0.09      0.04      0.06        75
          26       0.00      0.00      0.00         9
          30       0.00      0.00      0.00        54
          31       0.29      0.13      0.18       104
          32       0.00      0.00      0.00         2
          33       0.00      0.00      0.00        50
          61       0.00      0.00      0.00         1

    accuracy                           0.17       477
   macro avg       0.06      0.09      0.06       477
weighted avg       0.13      0.17      0.13       477



c:\Users\apkar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\apkar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
c:\Users\apkar\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, mo

In [ ]:
importances = model.feature_importances_

# Create a DataFrame for better visualization
feature_importance_df = pd.DataFrame({
    "Feature": xtr.columns,
    "Importance": importances
}).sort_values(by="Importance", ascending=False)

print(feature_importance_df)

In [ ]:
''' 
Percentile for entire dataset is considered, not just train --> Slight data leakage during state defining
Predefined state
unseen States appear in test dataset
High correlation/co-occurance between open and close --> Sigle feature category
'''